# proto-REINFORCE, Last-Minute Pricing, part 3: the backward pass

proto-REINFORCE learns to price the plane. Its forward pass goes from a table of bets to the average MONEY of 2,500
flights; parts 1 and 2 build it. Here the forward pass is given, finished, in a folded cell, and you build the
backward pass: a bet picked at random, its slope, its update, and at the end the whole loop. This notebook does
not need parts 1 and 2.

**How it works.** Run the cells from top to bottom and fill in the 🎯 cells: in each of them, replace every `...`
with your code. Every other cell is ready to run. A folded cell with a title holds plumbing: run it, and open it
only if you are curious. The 💡 hints are folded cells too: double-click one to open it, and open the next one
only if you are still stuck. After each task, a check cell tells you whether your code works. The luck of the
game is in the kit: your code never deals with dice.

Start with the three folded cells below: the plane's kit, the forward pass and the checks.

In [ ]:
#@title The plane's kit (run me first) { display-mode: "form" }
# Last-Minute Pricing's kit. Run this cell first; you do not need to read it.
import random                   # Python's random numbers: random.randint(0, 14) gives 0, 1, 2, ..., 14
from math import exp            # e to the power: exp(0.4) is 1.4918...
import math as _kit_math

SEATS = 5                       # the seats of the plane
DAYS = 4                        # the days to takeoff: DAY 1, 2, 3 and 4
PRICES = ['PREMIUM', 'STANDARD']
PREMIUM, STANDARD = 0, 1                # the columns of a table: the prices
DOLLARS = [5, 2]                  # a seat costs $5 at PREMIUM and $2 at STANDARD
FLIGHTS = 2500                  # the flights of every average
# The rows of a table: every situation a flight can be in, [seats left, day]. STANDARD sells at most 3 seats a
# day, so on DAY 2 at least 2 seats are left; on DAY 3 and DAY 4 anything from 1 to 5.
SITUATIONS = [[5, 1], [5, 2], [4, 2], [3, 2], [2, 2], [5, 3], [4, 3], [3, 3], [2, 3], [1, 3], [5, 4], [4, 4], [3, 4], [2, 4], [1, 4]]

# The plane of the arcade (levers.js and pricing.js): how many buyers come in a day, with their chances
_KIT_BUYERS = [[[0, 0.6], [1, 0.4]], [[1, 0.2], [2, 0.4], [3, 0.4]]]
_KIT_START_BETS = [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]
_KIT_BEST_LINE = 11.8392       # the best strategy's average money on the flights of update 240
_KIT_BEST = [0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 2]      # the best price of each row (2: either)
_KIT_LOSES = [0.0256, 0.224, 0.064, 0.96, 2.016, 2.08, 1.12, 0.16, 0.96, 1.2, 2.4, 2.4, 2.4, 1.6, 0.0]      # what the other price loses there, on average
_KIT_OFTEN = [1.0, 0.6, 0.4, 0.0, 0.0, 0.0, 0.36, 0.4, 0.24, 0.0, 0.0, 0.0, 0.312, 0.448, 0.24]      # the share of the best strategy's flights that are in that row
_KIT_FIRST_LUCK = 12                # the luck of the start
_KIT_LUCK_BASE = 2000              # update i of the loop flies the luck of seed 2000 + i
_KIT_PICKS = 1                       # the seed of Python's random for the loop's picks (play sets it)


def _kit_sell(seats_left, price, u):
    """One day of the plane on a given number u between 0 and 1 (pricing.js's sample): the seats sold and the
    money."""
    if isinstance(seats_left, bool) or seats_left not in (1, 2, 3, 4, 5) or isinstance(price, bool) or \
            price not in (0, 1):
        raise ValueError("sell needs 1 to 5 seats left and a price, 0 for PREMIUM or 1 for STANDARD; it got %r "
                         "seats left and the price %r" % (seats_left, price))
    running = 0
    buyers = _KIT_BUYERS[price][-1][0]
    for count, chance in _KIT_BUYERS[price]:
        running = running + chance           # the running chances, added as pricing.js adds them
        if u < running:
            buyers = count
            break
    seats_sold = min(buyers, seats_left)
    return seats_sold, DOLLARS[price] * seats_sold


# The luck: one stream of numbers between 0 and 1 (Mulberry32, the generator of the arcade), two numbers a day
_KIT_LUCK = {"seed": _KIT_FIRST_LUCK, "state": _KIT_FIRST_LUCK, "update": 0, "day": 0}


def _kit_next():
    s = (_KIT_LUCK["state"] + 0x6D2B79F5) & 0xFFFFFFFF
    _KIT_LUCK["state"] = s
    t = ((s ^ (s >> 15)) * (1 | s)) & 0xFFFFFFFF
    t = ((t + (((t ^ (t >> 7)) * (61 | t)) & 0xFFFFFFFF)) & 0xFFFFFFFF) ^ t
    return ((t ^ (t >> 14)) & 0xFFFFFFFF) / 4294967296


def spin():
    """The ball of a roulette: a number between 0 and 1, every number as likely as any other."""
    return _kit_next()


def sell(seats_left, price):
    """One day of the plane. With this many seats left and this price, luck decides how many buyers come, with the
    game's chances; they buy what is left. Gives the seats sold and the day's money:
        seats_sold, money = sell(5, STANDARD)"""
    if seats_left is Ellipsis or price is Ellipsis:
        print("⏳ sell waits for the seats left and a price: replace each ... with yours, then run the cell again.")
        return None, None
    seats_sold, money = _kit_sell(seats_left, price, _kit_next())
    _KIT_LUCK["day"] = _KIT_LUCK["day"] + 1
    if seats_sold == seats_left or _KIT_LUCK["day"] >= DAYS:
        # the flight is over: the luck of its unused days is skipped, so the next flight starts on its own numbers
        while _KIT_LUCK["day"] < DAYS:
            _kit_next()
            _kit_next()
            _KIT_LUCK["day"] = _KIT_LUCK["day"] + 1
        _KIT_LUCK["day"] = 0
    return seats_sold, money


def same_luck():
    """The luck starts again from its beginning: the flights that follow meet the same buyers as the flights of every
    average before them, so two tables of bets are compared on the same luck."""
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]
    _KIT_LUCK["day"] = 0


def new_luck():
    """New flights for the next update of proto-REINFORCE: the luck starts from a new seed, 2000 plus the number of
    the update."""
    _KIT_LUCK["update"] = _KIT_LUCK["update"] + 1
    _KIT_LUCK["seed"] = _KIT_LUCK_BASE + _KIT_LUCK["update"]
    _KIT_LUCK["state"] = _KIT_LUCK["seed"]
    _KIT_LUCK["day"] = 0


def _kit_set_luck(seed, update=0):
    """For the checks: the luck of this seed, from its beginning."""
    _KIT_LUCK["seed"] = seed
    _KIT_LUCK["state"] = seed
    _KIT_LUCK["update"] = update
    _KIT_LUCK["day"] = 0


def _kit_before_long():
    """play calls this before the loop: the luck of the start, and the same picks for everyone."""
    _kit_set_luck(_KIT_FIRST_LUCK)
    random.seed(_KIT_PICKS)


def _kit_after_long():
    """play calls this after the loop: the luck of the start again, for the cells above."""
    _kit_set_luck(_KIT_FIRST_LUCK)


def _kit_seed_picks(seed):
    """For the checks: Python's random from this seed."""
    random.seed(seed)


def _kit_save():
    """For the checks: the state of the luck and of Python's random."""
    return [dict(_KIT_LUCK), random.getstate()]


def _kit_restore(saved):
    """For the checks: the luck and Python's random as they were."""
    _KIT_LUCK.update(saved[0])
    random.setstate(saved[1])


def start_bets():
    """The starting table of bets, picked at random between -10 and 10: a fresh copy each time."""
    return copy_bets(_KIT_START_BETS)


def copy_bets(bets):
    """A copy of a table of bets: change the copy, and the table stays as it was."""
    copy = []
    for row in bets:
        copy.append(list(row))
    return copy


def _kit_situation(row):
    """DAY 1, 5 SEATS LEFT"""
    seats_left, day = SITUATIONS[row]
    return "DAY %d, %d %s LEFT" % (day, seats_left, "SEAT" if seats_left == 1 else "SEATS")


def show_situations():
    """Print the situations a flight can be in, one per row of a table."""
    print("row   situation             [seats left, day]")
    for row in range(len(SITUATIONS)):
        print("%3d   %-20s  %s" % (row, _kit_situation(row), SITUATIONS[row]))


def _kit_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _kit_is_table(table):
    if not isinstance(table, list) or len(table) != len(SITUATIONS):
        return False
    for row in table:
        if not isinstance(row, list) or len(row) != 2:
            return False
        for x in row:
            if not _kit_is_number(x):
                return False
    return True


def _kit_short(value):
    text = repr(value).replace("Ellipsis", "...")
    return text if len(text) <= 60 else text[:57] + "..."


def _kit_print_table(table, places, mark):
    head = "row   situation           " + "%10s%10s" % (PRICES[0], PRICES[1])
    if mark:
        head += "    likeliest"
    print(head)
    for row in range(len(SITUATIONS)):
        line = "%3d   %-20s" % (row, _kit_situation(row)) + "%10.*f%10.*f" % (places, table[row][0], places,
                                                                            table[row][1])
        if mark:
            line += "    " + PRICES[0 if table[row][0] > table[row][1] else 1]
        print(line)


def show_bets(bets):
    """Print a table of bets: a row per situation, a column per price, two decimals. Silent for None (a line above
    says why)."""
    if bets is None:
        return
    if not _kit_is_table(bets):
        print("⏳ No table of bets to show yet: show_bets needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(bets)))
        return
    _kit_print_table(bets, 2, False)


def show_chances(table):
    """Print a probability table (a policy): a row per situation, a column per price, two decimals, and the
    likeliest price of each row. Silent for None (a line above says why)."""
    if table is None:
        return
    if not _kit_is_table(table):
        print("⏳ No probability table to show yet: show_chances needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(table)))
        return
    _kit_print_table(table, 2, True)


def show_counts(counts):
    """Print a table of counts: a row per situation, a column per price, whole numbers."""
    if counts is None:
        return
    if not _kit_is_table(counts):
        print("⏳ No counts to show yet: show_counts needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(counts)))
        return
    _kit_print_table(counts, 0, False)


def show_roulette(chances, ball=None):
    """Print the roulette of a row of chances: each price's slot, starting where the slot before it ends, on a wheel
    of 50 marks from 0 to 1. With a ball, a ^ marks where it is."""
    ok = isinstance(chances, list) and len(chances) == len(PRICES)
    if ok:
        for x in chances:
            if not _kit_is_number(x) or x < 0:
                ok = False
    if not ok:
        print("⏳ No roulette to show yet: show_roulette needs a row of two chances. It got %s." % _kit_short(chances))
        return
    start = 0
    for a in range(len(PRICES)):
        end = start + chances[a]
        left = int(round(start * 50))
        right = int(round(end * 50))
        print("%-9s %4.2f to %4.2f  |%s%s" % (PRICES[a], start, end, " " * left, "#" * (right - left)))
        start = end
    print(" " * 25 + "0" + " " * 49 + "1")
    if ball is not None and _kit_is_number(ball) and 0 <= ball < 1:
        print(" " * 25 + " " * int(ball * 50) + "^ the ball, %.2f" % ball)


def _kit_dollars(x):
    """$5, $10.2596: whole dollars without cents, an average with every digit (at most 4 decimals)."""
    if x == int(x):
        text = "$%d" % abs(int(x))
    else:
        text = ("$%.4f" % abs(x)).rstrip("0")
    return "-" + text if x < 0 else text


def show_day(seats_sold, money):
    """Print one day of the plane: the seats sold and the day's money. Silent when there is no day yet (a line above
    says why)."""
    if seats_sold is None and money is None:
        return
    if not _kit_is_number(seats_sold) or not _kit_is_number(money):
        print("⏳ No day to show yet: show_day needs the seats sold and the day's money. It got %s and %s."
              % (_kit_short(seats_sold), _kit_short(money)))
        return
    print("Seats sold: %d. The day's money: %s." % (seats_sold, _kit_dollars(money)))


def show_money(label, money):
    """Print an amount of money, or a list of amounts, after a label: show_money("MONEY:", 12) prints MONEY: $12.
    A long list shows its first 10. Silent when there is nothing to show yet (a line above says why)."""
    if money is None:
        return
    if isinstance(money, list):
        parts = []
        for k in range(len(money)):
            if k == 10:
                parts.append("... (%s in all)" % format(len(money), ","))
                break
            if _kit_is_number(money[k]):
                parts.append(_kit_dollars(money[k]))
            else:
                parts.append(_kit_short(money[k]))
        print(label, ", ".join(parts) if parts else "none")
    elif _kit_is_number(money):
        print(label, _kit_dollars(money))
    else:
        print(label, _kit_short(money))


def show_value(label, value):
    """Print a value after a label: show_value("nabla =", -0.16). Silent for None (a line above says why)."""
    if value is None:
        return
    print(label, _kit_short(value))


def _kit_money(x):
    """$10.18"""
    text = "$%.2f" % abs(x)
    return "-" + text if x < 0 else text


def plot_rewards(points, best=_KIT_BEST_LINE, label="best strategy"):
    """The reward plot, as text: across, the number of updates; up, the average money of a flight after them. o is
    a point (one column for every few updates when there are many), and the row of dashes marks the best strategy's
    average money on the flights of the last update. Silent for None (a line above says why)."""
    if points is None:
        return
    ok = isinstance(points, list) and len(points) >= 2
    if ok:
        for x in points:
            if not _kit_is_number(x):
                ok = False
    if not ok:
        print("⏳ No reward plot yet: plot_rewards needs a list of at least two averages. It got %s." % _kit_short(points))
        return
    n = len(points) - 1                      # the updates
    columns = min(n + 1, 61)
    where = []                               # the update that each column shows
    for c in range(columns):
        where.append(int(round(c * n / (columns - 1))))
    values = []
    for i in where:
        values.append(points[i])
    low, high = min(values + [best]), max(values + [best])
    step = 0.1
    for size in (0.1, 0.2, 0.25, 0.5, 1, 2, 5, 10):
        step = size
        if (high - low) / size <= 18:
            break
    top = _kit_math.ceil(round(high / step, 9)) * step
    bottom = _kit_math.floor(round(low / step, 9)) * step
    rows = int(round((top - bottom) / step)) + 1
    grid = []
    for r in range(rows):
        grid.append([" "] * columns)
    best_row = int(round((top - best) / step))
    for c in range(columns):
        grid[best_row][c] = "-"
    for c in range(columns):
        r = int(round((top - values[c]) / step))
        grid[r][c] = "o"
    every = max(1, int(round(n / (columns - 1)))) if columns > 1 else 1
    print("The reward plot: the average money of a flight after each update" +
          (" (one o for every %d updates)" % every if every > 1 else ""))
    print()
    for r in range(rows):
        line = "%8s |" % _kit_money(top - r * step) + "".join(grid[r])
        if r == best_row:
            line += "  %s, %s" % (label, _kit_money(best))
        print(line.rstrip())
    print(" " * 9 + "+" + "-" * columns)
    ticks = 100 if n >= 300 else (50 if n >= 200 else (10 if n >= 40 else 1))
    marks = [" "] * (columns + 6)
    for t in range(0, n + 1, ticks):
        c = int(round(t * (columns - 1) / n)) if n else 0
        text = str(t)
        for j in range(len(text)):
            if c + j < len(marks):
                marks[c + j] = text[j]
    print(" " * 10 + "".join(marks).rstrip() + "   updates")
    print("first point %s, last point %s" % (_kit_money(points[0]), _kit_money(points[-1])))


def show_best(bets):
    """The best strategy, row by row, worked out backwards from DAY 4 with the game's chances, beside the likeliest
    price of a table of bets. Silent for None (a line above says why)."""
    if bets is None:
        return
    if not _kit_is_table(bets):
        print("⏳ No table of bets to compare yet: show_best needs %d rows of two numbers. It got %s."
              % (len(SITUATIONS), _kit_short(bets)))
        return
    print("best:  the best price in this situation, with the best prices after it")
    print("loses: what the other price loses here, on average, in dollars")
    print("often: the share of the best strategy's flights that are ever in this situation")
    print("yours: the likeliest price of your table")
    print()
    print("row   situation             best       loses   often   yours")
    same, matters = 0, 0
    for row in range(len(SITUATIONS)):
        yours = 0 if bets[row][0] > bets[row][1] else 1
        best = "either" if _KIT_BEST[row] == 2 else PRICES[_KIT_BEST[row]]
        mark = ""
        if _KIT_BEST[row] != 2:
            matters += 1
            if yours == _KIT_BEST[row]:
                same += 1
                mark = "  ✓"
            else:
                mark = "  ✗"
        print("%3d   %-20s  %-9s %6s   %4d%%   %-9s%s" % (row, _kit_situation(row), best, _kit_money(_KIT_LOSES[row]),
                                                       round(100 * _KIT_OFTEN[row]), PRICES[yours], mark))
    print()
    print("Your table's likeliest price is the best one in %d of the %d situations where the price matters." % (
        same, matters))


print("✅ Last-Minute Pricing's kit is ready.")

In [ ]:
#@title The forward pass of parts 1 and 2, given (run me too) { display-mode: "form" }
# The forward pass of parts 1 and 2, given: softmax, the roulette and their functions. You do not need to read this cell.


def softmax(row_of_bets):
    # A row of bets to a row of chances: e to the power of each bet, divided by their total, so that the chances
    # add up to 1 and a larger bet gets a larger chance. The largest bet is taken away first, as on page 3 of Old
    # Bessie: the chances stay the same, and exp never meets a number too large for it.
    largest = max(row_of_bets)
    powers = []
    for bet in row_of_bets:
        powers.append(exp(bet - largest))
    total = 0
    for power in powers:
        total = total + power
    chances = []
    for power in powers:
        chances.append(power / total)
    return chances


def probability_table(bets):
    # A table of bets to its probability table: softmax on every row.
    table = []
    for row in bets:
        table.append(softmax(row))
    return table



# Given: the ends of the slots of a roulette
def make_roulette(chances):
    # chances: a row of chances, one per price.
    # Gives the ends of the slots: a running total of the chances, such as [0.8, 1.0].
    ends = []
    running = 0                         # where the slots so far end: nowhere yet
    for chance in chances:
        running = running + chance      # this slot ends one chance further
        ends.append(running)            # remember where this slot ends
    return ends


# Given: the slot where the ball lands
def where_it_lands(ends, ball):
    # ends: the ends of the slots, from make_roulette. ball: a number between 0 and 1.
    # Gives the number of the first slot whose end is above the ball: 0 for PREMIUM, 1 for STANDARD.
    for slot in range(len(ends)):
        if ball < ends[slot]:           # the ball is below the end of this slot
            return slot
    return len(ends) - 1                # a ball at the very end of the wheel: the last slot


def row_of(seats_left, day):
    # Gives the row of the table for a situation: the place of [seats_left, day] in SITUATIONS.
    row = 0                                     # the row of the situation we look at
    for situation in SITUATIONS:                # situation[0]: its seats left, situation[1]: its day
        if situation[0] == seats_left and situation[1] == day:     # the seats left and the day both match
            return row
        row = row + 1                           # the next situation has the next row


def pick_action(policy, row):
    # policy: a probability table, one row per situation. row: the row of today's situation.
    # Gives a price picked at random with the chances of that row: 0 for PREMIUM, 1 for STANDARD.
    chances = policy[row]               # step 1: the row of chances
    ends = make_roulette(chances)       # step 2: its roulette
    ball = spin()                       # step 3: the ball
    return where_it_lands(ends, ball)   # step 4: where the ball lands


def two_days(policy):
    # policy: a probability table. Plays DAY 1 and DAY 2 of a flight, one after the other.
    # Gives the money of the two days, in a list.
    seats_left = SEATS                              # nobody has bought a seat yet
    row = row_of(seats_left, 1)                     # DAY 1: the row of the situation
    price = pick_action(policy, row)                # DAY 1: the policy picks a price
    seats_sold, money_1 = sell(seats_left, price)   # DAY 1: the plane's day
    seats_left = seats_left - seats_sold            # DAY 1: the seats sold are gone
    row = row_of(seats_left, 2)                     # DAY 2: the row of the situation
    price = pick_action(policy, row)                # DAY 2: the policy's price
    seats_sold, money_2 = sell(seats_left, price)   # DAY 2: the plane's day
    return [money_1, money_2]


def play_flight(policy):
    # policy: a probability table. Plays a flight from DAY 1 with 5 seats left, day after day.
    # Gives its rewards: the money of every day it sells on, in a list.
    rewards = []
    seats_left = SEATS
    for day in range(1, DAYS + 1):                  # DAY 1, 2, 3 and 4
        if seats_left > 0:                          # a FULL FLIGHT sells nothing more
            row = row_of(seats_left, day)           # the row of today's situation
            price = pick_action(policy, row)        # the policy picks a price
            seats_sold, money = sell(seats_left, price)     # the plane's day
            rewards.append(money)                   # remember the day's money
            seats_left = seats_left - seats_sold    # the seats sold today are gone
    return rewards


def cumulative_reward(rewards):
    # rewards: the money of every day of a flight, in a list.
    # Gives the flight's MONEY: the plain sum of its days, no discount.
    total = 0
    for money in rewards:
        total = total + money           # add the day's money
    return total                        # the total


def play_flights(policy, how_many):
    # Plays how_many flights with the same policy.
    # Gives the flights in a list, each flight the list of its rewards.
    flights = []
    for k in range(how_many):
        flight = play_flight(policy)    # one flight
        flights.append(flight)          # remember the flight
    return flights


def cumulative_rewards(flights):
    # flights: a list of flights, each the list of its rewards.
    # Gives the cumulative reward, the MONEY, of every flight, in a list.
    moneys = []
    for flight in flights:
        moneys.append(cumulative_reward(flight))    # the flight's MONEY
    return moneys                       # the list of every flight's MONEY


def average(numbers):
    # numbers: a list of numbers. Gives their average: their total divided by how many there are.
    total = 0
    for x in numbers:
        total = total + x               # add x to the total
    return total / len(numbers)         # the total divided by how many numbers there are


def average_money(bets):
    # bets: a table of bets. Gives the average MONEY of its policy over FLIGHTS flights, every digit kept.
    same_luck()                         # every average plays the same luck: the same buyers for every table of bets
    policy = probability_table(bets)            # the probability table of the bets
    flights = play_flights(policy, FLIGHTS)     # FLIGHTS flights with the policy
    moneys = cumulative_rewards(flights)        # the MONEY of every flight
    return average(moneys)              # their average, every digit


print("✅ The forward pass of parts 1 and 2 is ready: probability_table, play_flight, play_flights, cumulative_rewards, average_money.")

In [ ]:
#@title The checks of part 3 (run me too) { display-mode: "form" }
# The checks of notebook 2 and play. You do not need to read this cell.
# A check prints one short message and never stops the notebook.

_N2_TESTS = {'pick_bet': [{'about': 'pick 1 of the check', 'args': [], 'kind': 'numbers', 'want': [2, 0], 'tol': 0, 'setup': [['_kit_seed_picks', [1]]]}, {'about': 'pick 2 of the check', 'args': [], 'kind': 'numbers', 'want': [13, 0], 'tol': 0, 'setup': [['_kit_seed_picks', [2]]]}, {'about': 'pick 3 of the check', 'args': [], 'kind': 'numbers', 'want': [9, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [5]]]}, {'about': 'pick 4 of the check', 'args': [], 'kind': 'numbers', 'want': [3, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [8]]]}, {'about': 'pick 5 of the check', 'args': [], 'kind': 'numbers', 'want': [4, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [13]]]}, {'about': 'pick 6 of the check', 'args': [], 'kind': 'numbers', 'want': [2, 1], 'tol': 0, 'setup': [['_kit_seed_picks', [21]]]}], 'slope': [{'about': 'the bets at random, row 2 and PREMIUM, with a hop of 0.5, on the luck of the start', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 2, 0, 0.5], 'kind': 'number', 'want': -0.03, 'tol': 1e-09, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'the bets at random, row 11 and PREMIUM, on the luck of the start', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 11, 0, 0.01], 'kind': 'number', 'want': -0.16, 'tol': 1e-09, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'the bets at random, row 9 and PREMIUM, on the luck of the start', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 9, 0, 0.01], 'kind': 'number', 'want': 0.12, 'tol': 1e-09, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}, {'about': 'the bets at random, row 11 and STANDARD, on luck 2 of the check', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 11, 1, 0.01], 'kind': 'number', 'want': 0.24, 'tol': 1e-09, 'keep': [0], 'setup': [['_kit_set_luck', [2001]]]}, {'about': 'the bets at random, row 0 and PREMIUM, on the luck of the start', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 0, 0, 0.01], 'kind': 'number', 'want': 0.0, 'tol': 1e-09, 'keep': [0], 'setup': [['_kit_set_luck', [12]]]}], 'update_bet': [{'about': 'the bet by hand, row 11 and PREMIUM with nabla -0.16 and alpha 20', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 11, 0, -0.16, 20], 'kind': 'number', 'want': -3.1, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [-3.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}}, {'about': 'row 11 and STANDARD with nabla 0.24 and alpha 20', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 11, 1, 0.24, 20], 'kind': 'number', 'want': 4.6, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, 4.6], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}}, {'about': 'row 0 and PREMIUM with nabla 0 and alpha 20', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 0, 0, 0, 20], 'kind': 'number', 'want': 0.4, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}}, {'about': 'row 9 and PREMIUM with nabla 0.12 and alpha 25', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [-3.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 9, 0, 0.12, 25], 'kind': 'number', 'want': 3.4, 'tol': 1e-06, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [3.4, 0.3], [0.1, 5.3], [-3.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}}], 'proto_reinforce': [{'about': 'the bets at random, 2 updates with hop 0.01 and alpha 20, on the luck and the picks 2 of the check', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 2, 0.01, 20], 'kind': 'numbers', 'want': [10.2596, 10.4772, 10.3248], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[0.4, -8.8], [1.7, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [-2.4, -0.4], [-0.2, 0.3]]}, 'setup': [['_kit_before_long', []], ['_kit_seed_picks', [2]]]}, {'about': 'the bets at random, 2 updates with hop 0.01 and alpha 25, on the luck and the picks 2 of the check', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 2, 0.01, 25], 'kind': 'numbers', 'want': [10.2596, 10.498, 10.2956], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[0.4, -8.8], [2.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [-3.2, -0.4], [-0.2, 0.3]]}, 'setup': [['_kit_before_long', []], ['_kit_seed_picks', [2]]]}, {'about': 'the bets at random, 2 updates with hop 0.01 and alpha 20, on the luck and the picks 5 of the check', 'args': [[[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]], 2, 0.01, 20], 'kind': 'numbers', 'want': [10.2596, 10.3596, 10.6684], 'tol': 1e-09, 'table': {'arg': 0, 'want': [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, -2.1], [0.1, 5.3], [0.1, 4.6], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]}, 'setup': [['_kit_before_long', []], ['_kit_seed_picks', [5]]]}]}
_N2_SLIPS = {'pick_bet': [{'answers': [(1, 0), (0, 0), (2, 1), (1, 1), (2, 1), (1, 1)], 'hint': 'a row is one of the 15 situations: random.randint(0, 14) gives 0, 1, ..., 14'}, {'answers': [(3, 0), (14, 0), (10, 1), (4, 1), (5, 1), (3, 1)], 'hint': 'the rows count from 0: random.randint(0, 14) gives 0, 1, ..., 14'}, {'answers': [(2, 2), (13, 0), (9, 1), (3, 1), (4, 1), (2, 1)], 'hint': 'a column is one of the 2 prices: random.randint(0, 1) gives 0 or 1'}], 'slope': [{'answers': [-0.07, 0.6, -1.84, 0.96, 0.0], 'hint': 'raise the bet by the hop: hopped[row][price] + hop'}, {'answers': [0.0, 0.0, 0.0, 0.0, 0.0], 'hint': 'After is the average MONEY of the hopped bets: average_money(hopped)'}, {'answers': [-0.01, -0.0, 0.0, 0.0, 0.0], 'hint': 'divide the change by the hop: (after - before) / hop'}, {'answers': [0.03, 0.16, -0.12, -0.24, 0.0], 'hint': 'the change is after - before: when After is higher, the slope is positive'}], 'update_bet': [{'answers': [3.3, -5.0, 0.4, -2.6], 'hint': 'the average MONEY has to go up, so the update adds: new_bet = w + alpha * nabla'}, {'answers': [-0.06, 0.04, 0.4, 0.52], 'hint': 'alpha turns the small slope into a move you can see: new_bet = w + alpha * nabla'}, {'answers': [-3.2, 4.8, 0, 3.0], 'hint': 'the new bet starts from the bet as it is: new_bet = w + alpha * nabla'}], 'proto_reinforce': [{'answers': [[10.2596, -0.16, 0.08], [10.2596, -0.16, 0.08], [10.2596, -0.12, 0.24]], 'hint': 'a point is the average MONEY after the update: points.append(average_money(bets))'}, {'answers': [[10.2596, 10.31, 10.3616], [10.2596, 10.31, 10.3616], [10.2596, 10.31, 10.3616]], 'hint': 'each update moves its bet by alpha times nabla: update_bet(bets, row, price, nabla, alpha)'}, {'answers': [[10.2596, 10.3248], [10.2596, 10.2956], [10.2596, 10.6684]], 'hint': 'one point after every update: points.append(average_money(bets)) goes inside the loop'}]}
_N2_SUCCESS = {'pick_bet': "pick_bet works: a row from 0 to 14 and a column from 0 to 1, as Python's random picked them.", 'slope': 'slope works: the bet by hand, row 11 and PREMIUM, gives nabla = -0.16.', 'update_bet': 'update_bet works: the bet by hand moves from 0.10 to -3.10.', 'proto_reinforce': 'proto_reinforce works: its 2 updates of the check give the points $10.2596, $10.4772, $10.3248.'}
_N2_DEFAULT = {'pick_bet': 'row = random.randint(0, 14), then price = random.randint(0, 1)', 'slope': 'hopped[row][price] + hop, then after = average_money(hopped), then nabla = (after - before) / hop', 'update_bet': 'new_bet = w + alpha * nabla', 'proto_reinforce': 'row, price = pick_bet(), then nabla = slope(bets, row, price, hop), then update_bet(bets, row, price, nabla, alpha), then points.append(average_money(bets))'}
_N2_KEEP_HINT = {'pick_bet': 'pick_bet takes no inputs', 'slope': 'slope leaves the table of bets as it is: the hop goes into the copy, hopped', 'update_bet': 'update_bet changes only the bet of row and price: bets[row][price] = round(new_bet, 2)', 'proto_reinforce': 'proto_reinforce changes the table of bets it gets: update_bet(bets, row, price, nabla, alpha) on that table, with no copy'}
_N2_TASKS = {'pick_bet': 'task A', 'slope': 'task B', 'update_bet': 'task C', 'proto_reinforce': 'task D'}
_N2_NEEDS = {'pick_bet': [], 'slope': ['average_money'], 'update_bet': [], 'proto_reinforce': ['pick_bet', 'slope', 'update_bet', 'average_money']}
_N2_LONG = ['proto_reinforce']
_N2_NONE_HINT = {}
_N2_DICE_OF = 'spin'
_N2_GAPS = {}
_N2_WORKING = {}
_N2_DICE = {}


def _n2_num(x):
    """A number with at most 4 decimals: 154.3961, 2.14, 174."""
    if isinstance(x, bool) or not isinstance(x, (int, float)):
        return repr(x)
    if x != x or x in (float("inf"), float("-inf")):
        return repr(x)
    if isinstance(x, int):
        return str(x)
    text = ("%.4f" % x).rstrip("0").rstrip(".")
    if text == "-0":
        text = "0"
    if text == "0" and x != 0:
        text = "%.2g" % x
    return text


def _n2_text(value):
    """Numbers with at most 4 decimals, lists of them in brackets."""
    if isinstance(value, (list, tuple)):
        parts = []
        for x in value:
            parts.append(_n2_text(x))
        return "[" + ", ".join(parts) + "]"
    return _n2_num(value)


def _n2_short(value):
    """A value as text, cut short when it is long."""
    try:
        text = _n2_text(value)
    except Exception:
        text = "a value that cannot be printed"
    if len(text) > 72:
        text = text[:69] + "..."
    return text


def _n2_is_number(x):
    return not isinstance(x, bool) and isinstance(x, (int, float)) and x == x


def _n2_same(kind, got, want, tol):
    """True when got is the right answer want, within tol."""
    try:
        if kind == "number":
            return _n2_is_number(got) and abs(got - want) <= tol
        if kind == "action":
            return _n2_is_number(got) and got == want
        if kind in ("numbers", "table"):
            if not isinstance(got, (list, tuple)) or len(got) != len(want):
                return False
            for k in range(len(want)):
                if not _n2_same("numbers" if kind == "table" else "number", got[k], want[k], tol):
                    return False
            return True
    except Exception:
        return False
    return False


def _n2_like(got, answer, tol):
    """True when got has the shape and the numbers of a slip's answer, within tol: a slip may give a number where a
    list is due (return total), or a list of numbers where a list of lists is due."""
    if isinstance(answer, (list, tuple)):
        if not isinstance(got, (list, tuple)) or len(got) != len(answer):
            return False
        for k in range(len(answer)):
            if not _n2_like(got[k], answer[k], tol):
                return False
        return True
    return _n2_is_number(got) and _n2_is_number(answer) and abs(got - answer) <= tol


def _n2_copy(value):
    """A deep copy of lists of lists of numbers."""
    if isinstance(value, list):
        out = []
        for x in value:
            out.append(_n2_copy(x))
        return out
    return value


def _n2_dice(seed, first, count):
    """The dice of count runs of a seed from run first, rolled by the kit once per seed and kept."""
    rolled = _N2_DICE.get(seed)
    if rolled is None or len(rolled) < first + count:
        if len(_N2_DICE) > 3:
            _N2_DICE.clear()
        rolled = globals()[_N2_DICE_OF](seed, max(first + count, 10))
        _N2_DICE[seed] = rolled
    return rolled[first:first + count]


def _n2_args(test):
    """Fresh inputs for a test: copies of the lists, and the dice it names."""
    args = []
    for a in test["args"]:
        if isinstance(a, dict) and "dice" in a:
            args.append(_n2_dice(a["dice"][0], a["dice"][1], a["dice"][2]))
        elif isinstance(a, dict) and "run" in a:
            args.append(_n2_dice(a["run"][0], a["run"][1], 1)[0])
        else:
            args.append(_n2_copy(a))
    return args


def _n2_has_gap(fn):
    """True when the code of fn still holds a ... to fill in (comments do not count)."""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    if code in _N2_GAPS:
        return _N2_GAPS[code]
    # A gap such as "total = ..." stays in the compiled function; a gap such as "if ...:" only in the source, which
    # Jupyter and Colab keep for every cell
    gap = Ellipsis in code.co_consts
    if not gap:
        import inspect
        try:
            for line in inspect.getsource(fn).splitlines():
                if "..." in line.split("#")[0]:
                    gap = True
        except Exception:
            pass
    _N2_GAPS[code] = gap
    return gap


def _n2_holds_gap(value):
    """True when a value, or a list inside it, still holds a ... (a gap of a cell above)."""
    if value is Ellipsis:
        return True
    if isinstance(value, (list, tuple)):
        for x in value:
            if _n2_holds_gap(x):
                return True
    return False


def _n2_uses(fn, other):
    """True when the code of fn calls the function named other."""
    code = getattr(fn, "__code__", None)
    return code is not None and other in code.co_names


def _n2_key(name, seen=()):
    """The code of the function called name and of the functions it needs: a new version of any of them is checked
    again."""
    if name in seen:
        return ()
    fn = globals().get(name)
    key = [getattr(fn, "__code__", None)]
    for other in _N2_NEEDS.get(name, []):
        key.extend(_n2_key(other, seen + (name,)))
    return tuple(key)


def _n2_call(fn, args):
    """Run fn with its prints silenced: {"answer", "error" (None when the call went through), "line" (the line of
    your code where it happened)}. Never raises."""
    import contextlib, io, traceback
    run = {"answer": None, "error": None, "line": None}
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            run["answer"] = fn(*args)
    except Exception as error:
        run["error"] = "%s: %s" % (type(error).__name__, error)
        try:
            skip = (_n2_call.__code__.co_filename, globals()[_N2_DICE_OF].__code__.co_filename)
            for frame in traceback.extract_tb(error.__traceback__):
                if frame.filename not in skip and frame.line:
                    run["line"] = frame.line.strip()
        except Exception:
            run["line"] = None
    return run


def _n2_explain(run):
    """A hint for an error."""
    text = run["error"]
    if "NoneType" in text:
        return 'a value in that line is None: a function it calls gave nothing back. row_of gives None for a situation that is not in SITUATIONS: call it as row_of(seats_left, day)'
    if "ellipsis" in text:
        return "a ... is still in a function that this one calls: finish that function first"
    if "list indices must be integers" in text:
        return 'a place in [ ] must be a whole number: a row or a price, such as policy[row] for the row of a situation and chances[price] for one of its numbers'
    if "is not subscriptable" in text:
        return "that line uses [ ] on a single number: only a list has places to look up"
    if "is not iterable" in text:
        return ("a single number stands where a list was expected: check the names on the left of each = (update_bet "
                "changes the table itself and gives back only the new bet)")
    if 'sell needs' in text:
        return 'the plane got a number of seats or a price it does not know: check what pick_action gives back, and that seats_left goes down by the seats sold'
    if "ZeroDivisionError" in text:
        return "a division by zero: divide by a number that cannot be 0, such as the total or len(...)"
    if "NameError" in text:
        return "a name in that line is not defined: check its spelling, and run the cells above it first"
    if "can only concatenate" in text or "unsupported operand" in text:
        return "that line mixes a list and a number: take one number of the list with [ ] first"
    if "positional argument" in text:
        return "a function got more or fewer inputs than it takes: compare the call with the function's def line"
    if "out of range" in text:
        return "a place in [ ] is past the end of the list: count from 0, the last place is len(...) - 1"
    return "read that line again, compare it with the hints, run the function's cell, then this check again"


def _n2_failure(name, fn):
    """None when fn passes every test of its task, else (head, what, hint). The kit's state (its luck) is put back
    afterwards when the kit can save it."""
    save = globals().get("_kit_save")
    saved = save() if callable(save) else None
    try:
        return _n2_failure_of(name, fn)
    finally:
        if saved is not None:
            globals()["_kit_restore"](saved)


def _n2_failure_of(name, fn):
    tests = _N2_TESTS[name]
    for k in range(len(tests)):
        test = tests[k]
        args = _n2_args(test)
        kept = _n2_copy(args)
        for setup in test.get("setup", []):
            globals()[setup[0]](*setup[1])
        run = _n2_call(fn, args)
        head = "On " + test["about"]
        if run["error"] is not None:
            where = ""
            if run["line"]:
                where = ", in the line  %s " % run["line"]
            return (head, "%s stopped with an error, %s%s" % (name, run["error"], where), _n2_explain(run))
        got = run["answer"]
        if not _n2_same(test["kind"], got, test["want"], test["tol"]):
            if got is None:
                return (head, "%s gave None" % name, _N2_NONE_HINT.get(
                    name, "a function gives its answer with return: keep the return line at its end"))
            hint = _N2_DEFAULT[name]
            for slip in _N2_SLIPS[name]:
                answer = slip["answers"][k]
                if answer is not None and _n2_like(got, answer, test["tol"]):
                    hint = slip["hint"]
                    break
            return (head, "%s should give %s, and yours gives %s" % (name, _n2_short(test["want"]),
                                                                    _n2_short(got)), hint)
        for i in test.get("keep", []):
            if args[i] != kept[i]:
                return (head, "%s changed its input %s" % (name, _n2_short(kept[i])), _N2_KEEP_HINT[name])
        if test.get("table"):
            t = test["table"]
            if not _n2_same("table", args[t["arg"]], t["want"], test["tol"]):
                return (head, "after %s the table of bets should be %s, and it is %s" % (
                    name, _n2_short(t["want"]), _n2_short(args[t["arg"]])), _N2_KEEP_HINT[name])
    return None


def _n2_works(name):
    """True when the function called name exists, is finished, the functions it uses work, and it passes its tests.
    Remembered for each version of it."""
    fn = globals().get(name)
    if not callable(fn) or getattr(fn, "__code__", None) is None or _n2_has_gap(fn):
        return False
    for other in _N2_NEEDS.get(name, []):
        if _n2_uses(fn, other) and not _n2_works(other):
            return False
    if name not in _N2_TESTS:
        return True
    key = _n2_key(name)
    if key not in _N2_WORKING:
        _N2_WORKING[key] = _n2_failure(name, fn) is None
    return _N2_WORKING[key]


def _n2_waits(name, fn, cell):
    """One calm line and True when fn cannot run yet: it is not finished, or a function it uses does not work yet."""
    task = ""
    if name in _N2_TASKS:
        task = " (%s)" % _N2_TASKS[name]
    if _n2_has_gap(fn):
        print("⏳ %s is not finished yet%s: replace each ... with your code, run its cell, then %s again."
              % (name, task, cell))
        return True
    for other in _N2_NEEDS.get(name, []):
        if not _n2_uses(fn, other) or _n2_works(other):
            continue
        if other in _N2_TASKS:
            needed = globals().get(other)
            what = "does not work yet"
            if callable(needed) and _n2_has_gap(needed):
                what = "is not finished yet"
            print("⏳ %s uses your %s (%s), which %s. Get check_%s(%s) to ✅ first, then run %s again."
                  % (name, other, _N2_TASKS[other], what, other, other, cell))
            return True
        # a given function: it waits for the tasks it uses
        for deeper in _N2_NEEDS.get(other, []):
            if deeper in _N2_TASKS and not _n2_works(deeper):
                print("⏳ %s uses %s, which uses your %s (%s). Get check_%s(%s) to ✅ first, then run %s again."
                      % (name, other, deeper, _N2_TASKS[deeper], deeper, deeper, cell))
                return True
        print("⏳ %s uses %s, which does not work yet: run its cell again, then %s." % (name, other, cell))
        return True
    return False


def _n2_check(name, fn):
    """Print one message about fn: it works, it is not finished yet, or the first test it fails and a hint."""
    if not callable(fn):
        print("❌ check_%s needs your function itself: check_%s(%s), with no brackets after %s." % (
            name, name, name, name))
        return
    if _n2_waits(name, fn, "this check"):
        return
    failure = _n2_failure(name, fn)
    if failure is None:
        print("✅ " + _N2_SUCCESS[name])
        return
    head, what, hint = failure
    print("❌ %s: %s." % (head, what))
    print("   💡 Hint: " + hint + ".")


def _n2_safe_check(name, fn):
    try:
        _n2_check(name, fn)
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_pick_bet(fn):
    """Task a: the bet of several picks, each from a seed of Python's random that the check sets"""
    _n2_safe_check("pick_bet", fn)

def check_slope(fn):
    """Task b: the slope of several bets, each on luck the check deals"""
    _n2_safe_check("slope", fn)

def check_update_bet(fn):
    """Task c: the new bet of several updates, and the table of bets after each"""
    _n2_safe_check("update_bet", fn)

def check_proto_reinforce(fn):
    """Task d: the points and the table of bets after a few updates, on the luck and the picks the check sets"""
    _n2_safe_check("proto_reinforce", fn)


def play(fn, *inputs):
    """Runs one of your functions and gives its answer: play(average_money, bets) is
    average_money(bets). When the function is not finished yet, uses a function that does not work yet, gets
    None from a cell above or stops with an error, play prints one calm line and gives None, so the cells below
    still run."""
    import time
    try:
        if not callable(fn):
            print("⏳ play needs your function itself, as in play(average_money, bets).")
            return None
        name = getattr(fn, "__name__", "your function")
        if _n2_waits(name, fn, "this cell"):
            return None
        for x in inputs:
            if x is None:
                print("⏳ %s waits for a value that a cell above did not give yet (its ⏳ line says why)." % name)
                return None
            if _n2_holds_gap(x):
                print("⏳ %s waits for a value that still holds a ...: fill it in above, run its cell, then this cell."
                      % name)
                return None
        if name in _N2_LONG and not _n2_works(name):
            print("⏳ %s waits until it passes its check: get check_%s(%s) to ✅ first, then run this cell again."
                  % (name, name, name))
            return None
        long = name in _N2_LONG
        if long and callable(globals().get("_kit_before_long")):
            globals()["_kit_before_long"]()
        start = time.time()
        try:
            answer = fn(*inputs)
        except Exception as error:
            print("❌ %s stopped with an error, %s: %s. Its check says more." % (name, type(error).__name__, error))
            return None
        finally:
            if long and callable(globals().get("_kit_after_long")):
                globals()["_kit_after_long"]()
        seconds = time.time() - start
        if seconds > 5:
            print("⏱️ %s took %d seconds." % (name, round(seconds)))
        return answer
    except Exception as error:
        print("❌ play could not run (%s). Run the folded cells at the top again, then this cell." % error)
        return None

_N2_TABLE = [[0.8, 0.2], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.6, 0.4], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.4, 0.6], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8], [0.2, 0.8]]
_N2_START_BETS = [[0.4, -8.8], [0.1, -0.7], [-0.8, 0.5], [-0.4, 0.1], [-0.5, 0.4], [6.8, -5.0], [-0.5, -0.6], [6.9, 0.5], [-0.4, 0.0], [0.4, 0.3], [0.1, 5.3], [0.1, -0.2], [-0.9, 9.8], [0.8, -0.4], [-0.2, 0.3]]
_N2_READ_ROW = 8
_N2_MY = [0, 1, 5]


def _n2_ready(name, cell):
    """True when your function called name exists and passes its check; else one calm line."""
    fn = globals().get(name)
    if not callable(fn):
        print("⏳ %s is not there yet: run the cell of %s first, then %s." % (name, _N2_TASKS.get(name, name), cell))
        return False
    if _n2_waits(name, fn, cell):
        return False
    if not _n2_works(name):
        print("⏳ This cell waits until %s passes its check: get check_%s(%s) to ✅ first, then run %s again."
              % (name, name, name, cell))
        return False
    return True


def _n2_given_ready(names, cell):
    """True when the given functions called names exist; else one calm line."""
    for name in names:
        if not callable(globals().get(name)):
            print("⏳ %s is not there yet: run the folded cells at the top and the given cells above, then %s."
                  % (name, cell))
            return False
    return True


def _n2_policy_ready(policy, cell):
    """True when policy is a probability table without gaps; else one calm line."""
    if policy is None or _n2_holds_gap(policy):
        print("⏳ %s waits for a probability table: finish my_table first (warm-up 1), then run it again." % cell)
        return False
    return True


def check_table(table):
    """Warm-up 1: my_table, the made-up probability table."""
    try:
        if _n2_holds_gap(table):
            print("⏳ my_table is not finished yet (warm-up 1): replace each ... with a row of two chances, run its "
                  "cell, then this check again.")
            return
        if not isinstance(table, list) or len(table) != len(SITUATIONS):
            print("❌ my_table should be a list of %d rows, one per situation, and yours has %s." % (
                len(SITUATIONS), len(table) if isinstance(table, list) else _n2_short(table)))
            print("   💡 Hint: write one row per line, each in [ ] and followed by a comma, all inside the outer [ ].")
            return
        for r in range(len(SITUATIONS)):
            row = table[r]
            if not _n2_same("numbers", row, _N2_TABLE[r], 1e-9):
                print("❌ Row %d, %s, should be %s, and yours is %s." % (r, _kit_situation(r), _n2_text(_N2_TABLE[r]),
                                                                       _n2_short(row)))
                hint = "copy the row from the table above, two numbers in the order PREMIUM, STANDARD"
                if isinstance(row, list) and len(row) == 2 and _n2_same("numbers", [row[1], row[0]], _N2_TABLE[r],
                                                                        1e-9):
                    hint = "the numbers of a row go in the order of the columns: PREMIUM, STANDARD"
                print("   💡 Hint: " + hint + ".")
                return
        print("✅ my_table is the made-up probability table: %d rows, one per situation, and each row adds up to 1."
              % len(SITUATIONS))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_reading(chance, row):
    """Warm-up 2: a number and a row read from my_table."""
    try:
        if chance is Ellipsis or _n2_holds_gap(row):
            print("⏳ Warm-up 2 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        want_chance = _N2_TABLE[_N2_READ_ROW][STANDARD]
        want_row = _N2_TABLE[0]
        if not _n2_same("number", chance, want_chance, 1e-9):
            print("❌ chance should be %s, the chance of STANDARD in row %d, and yours is %s."
                  % (_n2_text(want_chance), _N2_READ_ROW, _n2_short(chance)))
            hint = "the row comes first, then the column: my_table[%d][STANDARD]" % _N2_READ_ROW
            if _n2_same("number", chance, _N2_TABLE[_N2_READ_ROW][PREMIUM], 1e-9):
                hint = "that is the other column of row %d: STANDARD is the column 1" % _N2_READ_ROW
            print("   💡 Hint: " + hint + ".")
            return
        if not _n2_same("numbers", row, want_row, 1e-9):
            print("❌ row should be row 0, %s, and yours is %s." % (_n2_text(want_row), _n2_short(row)))
            print("   💡 Hint: a row is the table with one [ ]: my_table[0].")
            return
        print("✅ Both right: my_table[%d][STANDARD] is %s, and my_table[0] is %s." % (
            _N2_READ_ROW, _n2_text(want_chance), _n2_text(want_row)))
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_day(seats_sold, money):
    """Warm-up 3: one day at STANDARD with 5 seats left, sell(5, STANDARD)."""
    try:
        if seats_sold is None and money is None:
            print("⏳ Warm-up 3 is not finished yet: replace each ... with your code, run its cell, then this check "
                  "again.")
            return
        for count, chance in _KIT_BUYERS[STANDARD]:
            if seats_sold == min(count, SEATS) and money == DOLLARS[STANDARD] * min(count, SEATS):
                print("✅ A day at STANDARD with 5 seats left: %d %s sold, %s." % (
                    seats_sold, "seat" if seats_sold == 1 else "seats", _kit_dollars(money)))
                return
        print("❌ A day at STANDARD sells 1, 2 or 3 seats at %s each; your day gave %s seats sold and the money %s."
              % (_kit_dollars(DOLLARS[STANDARD]), _n2_short(seats_sold), _n2_short(money)))
        hint = "call sell with the seats left first, then the price: sell(5, STANDARD)"
        if [seats_sold, money] in ([0, 0], [1, DOLLARS[PREMIUM]]):
            hint = "that is a day at PREMIUM: the price of this warm-up is STANDARD"
        print("   💡 Hint: " + hint + ".")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def check_my_bets(bets):
    """Warm-up 1 of part 2: the bets at random with the bet of row 0 and STANDARD set to 5."""
    try:
        r0, p0, v0 = _N2_MY
        if bets is None or _n2_holds_gap(bets):
            print("⏳ Warm-up 1 is not finished yet: replace the ... with your code, run its cell, then this check "
                  "again.")
            return
        if not _kit_is_table(bets):
            print("❌ my_bets should be a table of bets, %d rows of two numbers, and it is %s." % (
                len(SITUATIONS), _n2_short(bets)))
            print("   💡 Hint: start from my_bets = start_bets(), then change one bet.")
            return
        changed = []
        for r in range(len(SITUATIONS)):
            for p in range(2):
                if bets[r][p] != _N2_START_BETS[r][p]:
                    changed.append([r, p])
        if len(changed) == 0:
            print("⏳ my_bets is still the table at random: set the bet of row 0 and STANDARD to 5 in the cell of "
                  "warm-up 1, run it, then this check again.")
            return
        if changed == [[r0, p0]] and bets[r0][p0] == v0:
            print("✅ The bet of row 0 and STANDARD is 5: the chance of STANDARD on DAY 1 went from %.2f to %.2f." % (
                _n2_chances(_N2_START_BETS[r0])[p0], _n2_chances(bets[r0])[p0]))
            return
        if len(changed) > 1:
            print("❌ %d bets of my_bets differ from the table at random; warm-up 1 changes one." % len(changed))
            print("   💡 Hint: start again from my_bets = start_bets(), then change only my_bets[0][STANDARD].")
            return
        r, p = changed[0]
        if [r, p] != [r0, p0]:
            print("❌ You changed the bet of row %d and %s; warm-up 1 changes the bet of row 0 and STANDARD." % (
                r, PRICES[p]))
            print("   💡 Hint: the row comes first, then the column: my_bets[0][STANDARD] = 5.")
            return
        print("❌ The bet of row 0 and STANDARD is %s; warm-up 1 sets it to 5." % _n2_short(bets[r][p]))
        print("   💡 Hint: my_bets[0][STANDARD] = 5.")
    except Exception as error:
        print("❌ The check could not run (%s). Run the folded cells at the top again, then this cell." % error)


def _n2_chances(row):
    largest = max(row)
    powers = []
    for x in row:
        powers.append(exp(x - largest))
    total = 0
    for x in powers:
        total = total + x
    out = []
    for x in powers:
        out.append(x / total)
    return out


def throw_balls(chances, throws):
    """Throws the ball throws times on the roulette of a row of chances, with make_roulette and where_it_lands, and
    prints how often each price came."""
    try:
        if chances is None or _n2_holds_gap(chances):
            print("⏳ throw_balls waits for a row of chances: finish my_table first (warm-up 1), then run it again.")
            return
        if not _n2_given_ready(["make_roulette", "where_it_lands"], "this cell"):
            return
        ends = make_roulette(chances)
        counts = [0, 0]
        for k in range(throws):
            slot = where_it_lands(ends, spin())
            counts[slot] = counts[slot] + 1
        print("In %s throws:  PREMIUM %d times, STANDARD %d times" % (format(throws, ","), counts[0], counts[1]))
        print("Their shares:     PREMIUM %.2f, STANDARD %.2f" % (counts[0] / throws, counts[1] / throws))
        print("The chances:      PREMIUM %.2f, STANDARD %.2f" % (chances[0], chances[1]))
    except Exception as error:
        print("❌ throw_balls could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_picks(policy, picks):
    """Your pick_action, picks times in each row of a probability table: how often each price came, and the
    shares."""
    try:
        if not _n2_policy_ready(policy, "show_picks"):
            return
        if not _n2_ready("pick_action", "this cell"):
            return
        counts = []
        shares = []
        for r in range(len(SITUATIONS)):
            counts.append([0, 0])
            for k in range(picks):
                price = pick_action(policy, r)
                counts[r][price] = counts[r][price] + 1
            shares.append([counts[r][0] / picks, counts[r][1] / picks])
        print("Your pick_action, %s times in each row: the shares of each price, to set beside the probability table"
              % format(picks, ","))
        show_chances(shares)
    except Exception as error:
        print("❌ show_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def count_picks(how_many):
    """Your pick_bet, how_many times: how often each bet came."""
    try:
        if not _n2_ready("pick_bet", "this cell"):
            return
        counts = []
        for r in range(len(SITUATIONS)):
            counts.append([0, 0])
        for k in range(how_many):
            row, price = pick_bet()
            counts[row][price] = counts[row][price] + 1
        print("Your pick_bet, %s times: how often each bet came" % format(how_many, ","))
        show_counts(counts)
    except Exception as error:
        print("❌ count_picks could not run (%s). Run the folded cells at the top again, then this cell." % error)


def show_flight(policy):
    """Plays one flight with your row_of and pick_action and the kit's sell, and prints it day by day."""
    try:
        if not _n2_policy_ready(policy, "show_flight"):
            return
        if not _n2_ready("row_of", "this cell") or not _n2_ready("pick_action", "this cell"):
            return
        seats_left = SEATS
        total = 0
        print("day     situation             price     sold   money")
        for day in range(1, DAYS + 1):
            if seats_left > 0:
                row = row_of(seats_left, day)
                price = pick_action(policy, row)
                seats_sold, money = sell(seats_left, price)
                print("DAY %d   %-20s  %-8s  %4d   %5s" % (day, _kit_situation(row), PRICES[price], seats_sold,
                                                          _kit_dollars(money)))
                total = total + money
                seats_left = seats_left - seats_sold
        if seats_left == 0:
            print("FULL FLIGHT!   MONEY %s" % _kit_dollars(total))
        else:
            print("TAKEOFF! %d EMPTY %s.   MONEY %s" % (seats_left, "SEAT" if seats_left == 1 else "SEATS",
                                                       _kit_dollars(total)))
    except Exception as error:
        print("❌ show_flight could not run (%s). Run the folded cells at the top again, then this cell." % error)

print("✅ The checks of part 3 are ready.")

## 1. The forward pass, given

The plane: 5 seats, 4 days to takeoff, one price a day, PREMIUM at \$5 a seat or STANDARD at \$2. A table has one
row per situation, `[seats left, day]` (`show_situations()` lists the 15 of them), and one column per price,
`PREMIUM` (0) and `STANDARD` (1). The folded cell above holds the forward pass. This notebook calls three of its
functions:

- `start_bets()`: a fresh copy of the table of bets at random, between -10 and 10.
- `probability_table(bets)`: softmax on every row of a table of bets, so a larger bet gets a larger chance.
- `average_money(bets)`: the average MONEY of `FLIGHTS` flights with the policy of a table of bets, every digit
  kept. Every call plays the same flights, so two tables of bets are compared on the same luck.

In [ ]:
# The table of bets at random, its probability table and its average MONEY
bets = start_bets()
show_chances(probability_table(bets))
print()
show_money("The average MONEY of the bets at random:", play(average_money, bets))

**Reading the result.** \$10.2596: the first point of the reward plot. The backward pass raises it one bet at a time.

## 2. A bet at random

Each update of proto-REINFORCE picks one bet at random, every bet as likely as any other: a row, the situation,
and a column, the price. Python's `random.randint(0, 14)` gives 0, 1, 2, ..., 14, each as likely as the others:
a row. `random.randint(0, 1)` gives a column. Run the next cell a few times.

In [ ]:
# A row at random, then a column at random
row = random.randint(0, 14)
price = random.randint(0, 1)
print("The bet picked at random: row", row, "and", PRICES[price])

### 🎯 Task A. A bet at random

Write `pick_bet()`: a row at random, then a column at random, given back together. It takes no inputs.

In [ ]:
# ✅ SOLUTION, task A: a bet at random
def pick_bet():
    # Gives a bet picked at random, every bet as likely as any other: its row, the situation, and its column, the
    # price.
    row = random.randint(0, 14)         # a row at random, one of the 15 situations
    price = random.randint(0, 1)        # a column at random, one of the 2 prices
    return row, price

In [ ]:
#@title 💡 Hint 1 for task A (double-click to open) { display-mode: "form" }
# The two lines of the cell above: random.randint(0, 14) for one of the 15 rows, random.randint(0, 1) for one
# of the 2 columns.

In [ ]:
#@title 💡 Hint 2 for task A (double-click to open) { display-mode: "form" }
#     row = random.randint(0, 14)
#     price = random.randint(0, 1)

In [ ]:
# Check task A
check_pick_bet(pick_bet)

In [ ]:
# Your pick_bet, 3,000 times
count_picks(3000)

**Reading the result.** Each of the 30 bets comes about 100 times in 3,000 picks: every bet is as likely as any
other.

## 3. Before and After

Take the bet of PREMIUM on DAY 4 with 4 seats left, row 11. The bets at random make PREMIUM the likelier price
there, with a chance of 0.57. proto-REINFORCE asks how the average MONEY changes when that bet rises a little, by
a hop of 0.01. Before is the average MONEY of the bets as they are. After is the average MONEY with that one bet a
hop higher, on the same luck.

💭 **Think first.** On the last day, PREMIUM sells at most one seat and STANDARD one to three. Will a hop up of
this bet raise the average MONEY or lower it? And how many of the 2,500 flights do you expect to change?

In [ ]:
# A bet by hand: PREMIUM on DAY 4 with 4 seats left, on the luck of the start
hop = 0.01                                      # the hop: a small change of one bet, as on page 3 of Old Bessie
bets = start_bets()                             # the table of bets at random
row, price = 11, PREMIUM                        # row 11 is DAY 4 with 4 seats left
hopped = copy_bets(bets)                        # a copy: the table itself stays as it is
hopped[row][price] = hopped[row][price] + hop   # one hop higher

before = play(average_money, bets)              # Before: the bets as they are
after = play(average_money, hopped)             # After: one bet a hop higher, on the same luck
show_money("Before:", before)
show_money("After: ", after)

**Reading the result.** Lower. Only 1 of the 2,500 flights plays another price after the hop, and the average moves from \$10.2596 to \$10.258: a change of -\$0.0016.

The slope, $\nabla$ (nabla), is the change divided by the hop, to the hundredth:

$$\nabla = \frac{\text{After} - \text{Before}}{\text{hop}} = \frac{-0.0016}{0.01} = -0.16$$

A small number: the money of a flight is a few dollars, and a hop changes few flights. On page 3 of Old Bessie
the first nabla was 174. A negative slope says: lower this bet.

### 🎯 Task B. The slope

Write `slope(bets, row, price, hop)`: Before, After with the bet a hop higher in a copy of the bets, and the
change divided by the hop.

In [ ]:
# ✅ SOLUTION, task B: the slope, nabla
def slope(bets, row, price, hop):
    # How fast the average MONEY rises when the bet of row and price rises, measured with a hop.
    # Gives nabla, to the hundredth.
    before = average_money(bets)                # Before: the bets as they are
    hopped = copy_bets(bets)                    # a copy: the table itself stays as it is
    hopped[row][price] = hopped[row][price] + hop       # the bet, one hop higher
    after = average_money(hopped)               # After: the same luck, with the hopped bets
    nabla = (after - before) / hop              # the change divided by the hop
    return round(nabla, 2)

In [ ]:
#@title 💡 Hint 1 for task B (double-click to open) { display-mode: "form" }
# The cell above, inside a function: one hop added to the bet of the copy, After from average_money of the
# copy, then the change divided by the hop.

In [ ]:
#@title 💡 Hint 2 for task B (double-click to open) { display-mode: "form" }
#     hopped[row][price] = hopped[row][price] + hop
#     after = average_money(hopped)
#     nabla = (after - before) / hop

In [ ]:
# Check task B
check_slope(slope)

In [ ]:
# The slope of the bet by hand
nabla = play(slope, bets, row, price, hop)
show_value("nabla =", nabla)

## 4. The update

The update moves the bet by $\alpha$ times the slope, so that the average MONEY goes up:

$$w = w + \alpha \nabla$$

💭 **Think first.** On page 3 of Old Bessie, $\alpha$ was 0.01. With this nabla, how far would $\alpha = 0.01$ move
the bet?

In [ ]:
#@title The answer (double-click to open) { display-mode: "form" }
# alpha 0.01 would move the bet by 0.01 times -0.16, that is -0.0016: in cents, nothing. So on the plane alpha
# is 20, which turns this slope into a move of -3.2.

On the plane $\alpha = 20$: $w = 0.10 + 20 \times (-0.16) = -3.10$.

### 🎯 Task C. The update of one bet

Write `update_bet(bets, row, price, nabla, alpha)`: the new bet, $w + \alpha \nabla$. The given lines put it into
the table, in cents, and give it back.

In [ ]:
# ✅ SOLUTION, task C: the update of one bet
def update_bet(bets, row, price, nabla, alpha):
    # The update of one bet, w = w + alpha nabla, in cents. Changes the table of bets and gives the new bet.
    w = bets[row][price]                        # the bet as it is
    new_bet = w + alpha * nabla                 # a positive nabla raises the bet, a negative one lowers it
    bets[row][price] = round(new_bet, 2)        # into the table, in cents
    return bets[row][price]

In [ ]:
#@title 💡 Hint 1 for task C (double-click to open) { display-mode: "form" }
# w is the bet as it is, alpha the step and nabla the slope: the new bet is w plus alpha times nabla.

In [ ]:
#@title 💡 Hint 2 for task C (double-click to open) { display-mode: "form" }
#     new_bet = w + alpha * nabla

In [ ]:
# Check task C
check_update_bet(update_bet)

In [ ]:
# The update of that bet, then the same 2,500 flights again
alpha = 20                                      # alpha: turns a small slope into a move of a few units
bets = start_bets()                             # the table of bets at random
new_bet = play(update_bet, bets, row, price, nabla, alpha)
if new_bet is not None:                         # once update_bet works
    show_value("The new bet:", new_bet)
    print()
    show_chances(probability_table(bets))
    print()
    show_money("The average MONEY after the update:", play(average_money, bets))

**Reading the result.** PREMIUM's chance on DAY 4 with 4 seats left falls from 0.57 to 0.05, and the average MONEY of
the same flights goes from \$10.2596 to \$10.494.

## 5. proto-REINFORCE

Now the whole loop. Each update plays new flights, picks a bet at random, measures its slope, updates the bet,
and adds one point to the reward plot: the average MONEY after the update.

### 🎯 Task D. The loop

Write `proto_reinforce(bets, updates, hop, alpha)`. Its first point and the first line of each update are given:
`new_luck()` deals the update new flights. Each line you write calls one of your functions.

In [ ]:
# ✅ SOLUTION, task D: proto-REINFORCE on the plane
def proto_reinforce(bets, updates, hop, alpha):
    # bets: the table of bets, changed by every update. updates: how many updates.
    # Gives the points of the reward plot: the average MONEY before the first update, then after each update.
    points = [average_money(bets)]              # the first point: the bets as they are
    for update in range(updates):
        new_luck()                              # new flights for this update
        row, price = pick_bet()                         # a bet at random
        nabla = slope(bets, row, price, hop)            # the backward pass: the hop and the slope
        update_bet(bets, row, price, nabla, alpha)      # the update: w = w + alpha nabla
        points.append(average_money(bets))              # one more point: the average MONEY after the update
    return points

In [ ]:
#@title 💡 Hint 1 for task D (double-click to open) { display-mode: "form" }
# Your functions, in the order of an update: pick_bet for the bet, slope for nabla, update_bet for the update,
# average_money for the point.
# update_bet changes the table itself, so its line is the call alone.

In [ ]:
#@title 💡 Hint 2 for task D (double-click to open) { display-mode: "form" }
#         row, price = pick_bet()
#         nabla = slope(bets, row, price, hop)
#         update_bet(bets, row, price, nabla, alpha)
#         points.append(average_money(bets))

In [ ]:
# Check task D
check_proto_reinforce(proto_reinforce)

### 240 updates

💭 **Think first.** On DAY 4, the seats that are not sold fly empty. The bets at random play PREMIUM on DAY 4 with 2 seats left (a
chance of 0.77) and STANDARD on DAY 3 with 2 seats left (0.60). Which price do you expect there after the updates?

The next cell plays 240 updates from the bets at random, each with three averages of 2,500 flights: under a minute on Colab.
There are 30 bets, so each gets about 8 updates.

In [ ]:
# 240 updates of proto-REINFORCE on the plane, from the bets at random (under a minute on Colab)
bets = start_bets()
points = play(proto_reinforce, bets, 240, hop, alpha)
plot_rewards(points)

**Reading the result.** The first point is the start, \$10.26, and after update 240 the average is \$11.75. The row of
dashes is the best strategy's average MONEY on the flights of update 240, \$11.84. In 216 of the 240 updates nabla was 0:
the hop changed the money of no flight. A hop of 0.01 moves a chance by at most 0.0025, by less when the row's
chances are near 0 and 1, and a rare row has few flights to change. So the points climb in steps, and every
update plays new flights, which makes them wobble.

In [ ]:
# The probability table after the updates
if points is not None:                          # once the loop above has run
    show_chances(probability_table(bets))

### The best strategy

The kit works the best strategy out exactly, backwards from DAY 4 with the game's chances. It is in no table of
bets: proto-REINFORCE never sees it.

In [ ]:
# The best strategy beside your table's likeliest prices
if points is not None:                          # once the loop above has run
    show_best(bets)

**Reading the result.** The best strategy expects \$11.77 a flight. In the four situations where flights often go
and the price matters most, your table now plays the best price: STANDARD on DAY 4 with 3 or 2 seats left, since
an empty seat earns nothing at takeoff, and on DAY 3 with 4 seats left; PREMIUM on DAY 3 with 2 seats left, since
two days remain to find buyers at \$5. It still differs from the best strategy on DAY 2 with 3 seats left, DAY 2 with 2 seats left and DAY 3 with 5 seats left, situations its own flights almost never reach (fewer than 1 in 1,000): proto-REINFORCE learns where its own flights go. It also differs on DAY 2 with 4 seats left, where the other price loses only \$0.06. Your table expects \$11.73 a flight, up from \$10.35 at the start.

## What you built

- **The forward pass**, given: `average_money` gives the average MONEY of 2,500 flights of the policy of a table
  of bets.
- **A bet at random**: `pick_bet` picks a row and a column.
- **The backward pass**: `slope` raises one bet by a hop of 0.01, plays the same flights again and gives nabla.
- **The update**: `update_bet` moves the bet, $w = w + \alpha \nabla$, with $\alpha = 20$.
- **The loop**: `proto_reinforce` repeats it, one bet picked at random each time, and gives the points of the
  reward plot.

The same algorithm on Old Bessie: [part 1, runs and cumulative rewards](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_1a_old_bessie_runs_and_cumulative_rewards_solutions.ipynb), [part 2, the forward pass](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_1b_old_bessie_forward_pass_solutions.ipynb) and
[part 3, the backward pass](https://colab.research.google.com/github/eth-ainit-hs26/w3-cx-public/blob/main/03_cx_proto_reinforce_1c_old_bessie_backward_pass_solutions.ipynb).